## Ingest Dimension Data into Bronze Layer

In [1]:
import duckdb
import pandas as pd

#### Connect database

In [2]:
# Connect database
try:
    con = duckdb.connect("../data/finance_analytics_light_raw.duckdb")
    print("✅Successfully connected to the database")
except ValueError as e:
    print(f"Failed connected to the database{e}")

✅Successfully connected to the database


#### Show schema

In [3]:
# Check All Schema
con.sql(
    """
    SHOW SCHEMAS
    """
)

┌─────────────────────────────┬─────────────┬─────────┐
│        database_name        │ schema_name │ current │
│           varchar           │   varchar   │ boolean │
├─────────────────────────────┼─────────────┼─────────┤
│ finance_analytics_light_raw │ bronze      │ false   │
│ finance_analytics_light_raw │ gold        │ false   │
│ finance_analytics_light_raw │ main        │ true    │
│ finance_analytics_light_raw │ silver      │ false   │
└─────────────────────────────┴─────────────┴─────────┘

In [4]:
# Check All Tables from main schema
con.sql(
    """
    SHOW TABLES FROM finance_analytics_light_raw.main
    """
)

┌────────────────────────────┐
│            name            │
│          varchar           │
├────────────────────────────┤
│ raw__account_mapping       │
│ raw__accounts_light        │
│ raw__geography_light       │
│ raw__gl_transactions_light │
│ raw__stores_light          │
└────────────────────────────┘

#### Accounts

In [5]:
# Check raw__accounts_light
con.sql(
    """
    SELECT
    *
    FROM main.raw__accounts_light
    """
)

┌────────────────┬────────────────────┬───────────────────┬──────────┐
│ account_number │    account_name    │   account_type    │ currency │
│    varchar     │      varchar       │      varchar      │ varchar  │
├────────────────┼────────────────────┼───────────────────┼──────────┤
│ 3000           │ Product Sales      │ Revenue           │ NOK      │
│ 4000           │ Cost of Goods Sold │ COGS              │ NOK      │
│ 5000           │ Salaries & Wages   │ Operating Expense │ NOK      │
│ 5100           │ Rent Expense       │ Operating Expense │ NOK      │
│ 5200           │ Marketing Expense  │ Operating Expense │ NOK      │
│ 7000           │ Interest Expense   │ Financial         │ NOK      │
└────────────────┴────────────────────┴───────────────────┴──────────┘

In [6]:
# Create bronze accounts table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE bronze.accounts AS 
        SELECT 
            account_number, 
            account_name, 
            account_type, 
            currency
        FROM raw__accounts_light
        """
    )
    print("✅ Created bronze accounts")
except ValueError as error:
    print(f"Failed create bronze table accounts, {error}")

✅ Created bronze accounts


In [7]:
# Check bronze accounts table
con.sql(
    """
    SELECT
    *
    FROM bronze.accounts
    """
)

┌────────────────┬────────────────────┬───────────────────┬──────────┐
│ account_number │    account_name    │   account_type    │ currency │
│    varchar     │      varchar       │      varchar      │ varchar  │
├────────────────┼────────────────────┼───────────────────┼──────────┤
│ 3000           │ Product Sales      │ Revenue           │ NOK      │
│ 4000           │ Cost of Goods Sold │ COGS              │ NOK      │
│ 5000           │ Salaries & Wages   │ Operating Expense │ NOK      │
│ 5100           │ Rent Expense       │ Operating Expense │ NOK      │
│ 5200           │ Marketing Expense  │ Operating Expense │ NOK      │
│ 7000           │ Interest Expense   │ Financial         │ NOK      │
└────────────────┴────────────────────┴───────────────────┴──────────┘

#### Geography

In [8]:
# Check raw__geography_light
con.sql(
    """
    SELECT 
    *
    FROM raw__geography_light
    """
)

┌────────────┬─────────┬──────────┐
│ store_code │ country │  region  │
│  varchar   │ varchar │ varchar  │
├────────────┼─────────┼──────────┤
│ NO001      │ Norway  │ East     │
│ NO002      │ Norway  │ West     │
│ NO003      │ Norway  │ Mid      │
│ SE001      │ Sweden  │ East     │
│ SE002      │ Sweden  │ West     │
│ EC_NO      │ Norway  │ National │
│ EC_SE      │ Sweden  │ National │
└────────────┴─────────┴──────────┘

In [9]:
# Create bronze geography table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE bronze.geography AS 
        SELECT 
            store_code, 
            country, 
            region 
        FROM raw__geography_light
        """
    )
    print("✅ Created bronze geography")
except ValueError as error:
    print(f"Failed create bronze table geography, {error}")

✅ Created bronze geography


In [10]:
# Check bronze geography table
con.sql(
    """
    SELECT 
    *
    FROM bronze.geography
    """
)

┌────────────┬─────────┬──────────┐
│ store_code │ country │  region  │
│  varchar   │ varchar │ varchar  │
├────────────┼─────────┼──────────┤
│ NO001      │ Norway  │ East     │
│ NO002      │ Norway  │ West     │
│ NO003      │ Norway  │ Mid      │
│ SE001      │ Sweden  │ East     │
│ SE002      │ Sweden  │ West     │
│ EC_NO      │ Norway  │ National │
│ EC_SE      │ Sweden  │ National │
└────────────┴─────────┴──────────┘

#### General ledger transactions

In [11]:
# Check raw__gl_transactions_light
con.sql(
    """
    SELECT 
    *
    FROM raw__gl_transactions_light
    """
)

┌────────────────┬──────────────────┬────────────┬────────────────┬──────────────┬──────────┬─────────────────┬──────────────────────────────────────────────┐
│ transaction_id │ transaction_date │ store_code │ account_number │ amount_local │ currency │ document_number │                 description                  │
│     int64      │       date       │  varchar   │     int32      │    double    │ varchar  │     varchar     │                   varchar                    │
├────────────────┼──────────────────┼────────────┼────────────────┼──────────────┼──────────┼─────────────────┼──────────────────────────────────────────────┤
│              1 │ 2025-03-12       │ SE001      │           4000 │     -7149.34 │ NOK      │ DOC-2025-000001 │ Cost of Goods Sold - Recently thought month. │
│              2 │ 2025-09-24       │ SE001      │           4000 │     -6023.02 │ NOK      │ DOC-2025-000002 │ Cost of Goods Sold - Per send.               │
│              3 │ 2026-05-23       │ SE001   

In [12]:
# Create bronze gl_transactions table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE bronze.gl_transactions AS 
        SELECT 
            transaction_id , 
            transaction_date , 
            store_code  ,
            account_number ,
            amount_local ,
            currency ,
            document_number ,
            description                  
        FROM raw__gl_transactions_light
        """
    )
    print("✅ Created bronze gl_transactions")
except ValueError as error:
    print(f"Failed create bronze table gl_transactions, {error}")

✅ Created bronze gl_transactions


In [13]:
# Check bronze gl_transactions table
con.sql(
    """
    SELECT 
    *
    FROM bronze.gl_transactions
    """
)

┌────────────────┬──────────────────┬────────────┬────────────────┬──────────────┬──────────┬─────────────────┬──────────────────────────────────────────────┐
│ transaction_id │ transaction_date │ store_code │ account_number │ amount_local │ currency │ document_number │                 description                  │
│     int64      │       date       │  varchar   │     int32      │    double    │ varchar  │     varchar     │                   varchar                    │
├────────────────┼──────────────────┼────────────┼────────────────┼──────────────┼──────────┼─────────────────┼──────────────────────────────────────────────┤
│              1 │ 2025-03-12       │ SE001      │           4000 │     -7149.34 │ NOK      │ DOC-2025-000001 │ Cost of Goods Sold - Recently thought month. │
│              2 │ 2025-09-24       │ SE001      │           4000 │     -6023.02 │ NOK      │ DOC-2025-000002 │ Cost of Goods Sold - Per send.               │
│              3 │ 2026-05-23       │ SE001   

#### Stores

In [14]:
# Check raw__stores_light
con.sql(
    """
    SELECT 
    *
    FROM raw__stores_light
    """
)

┌────────────┬───────────────────┬────────────┐
│ store_code │    store_name     │ store_type │
│  varchar   │      varchar      │  varchar   │
├────────────┼───────────────────┼────────────┤
│ NO001      │ NRG Oslo City     │ Store      │
│ NO002      │ NRG Bergen        │ Store      │
│ NO003      │ NRG Trondheim     │ Store      │
│ SE001      │ NRG Stockholm     │ Store      │
│ SE002      │ NRG Gothenburg    │ Store      │
│ EC_NO      │ NRG Online Norway │ Ecom       │
│ EC_SE      │ NRG Online Sweden │ Ecom       │
└────────────┴───────────────────┴────────────┘

In [15]:
# Create bronze stores table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE bronze.stores AS 
        SELECT 
            store_code,
            store_name,
            store_type
        FROM raw__stores_light
        """
    )
    print("✅ Created bronze stores")
except ValueError as error:
    print(f"Failed create bronze table stores, {error}")

✅ Created bronze stores


In [16]:
# Check bronze stores table
con.sql(
    """
    SELECT 
    *
    FROM bronze.stores
    """
)

┌────────────┬───────────────────┬────────────┐
│ store_code │    store_name     │ store_type │
│  varchar   │      varchar      │  varchar   │
├────────────┼───────────────────┼────────────┤
│ NO001      │ NRG Oslo City     │ Store      │
│ NO002      │ NRG Bergen        │ Store      │
│ NO003      │ NRG Trondheim     │ Store      │
│ SE001      │ NRG Stockholm     │ Store      │
│ SE002      │ NRG Gothenburg    │ Store      │
│ EC_NO      │ NRG Online Norway │ Ecom       │
│ EC_SE      │ NRG Online Sweden │ Ecom       │
└────────────┴───────────────────┴────────────┘

### Account mapping


In [17]:
# Paths
excel_path = "../data/account_mapping_light_clean.xlsx"
df_account_mapping = pd.read_excel(excel_path)
df_account_mapping

,AccountNumber,AccountName,PLLine,StatementType,SortOrder,Notes
0,3000,Product Sales,Revenue,P&L,10,NaN
1,3010,Online Sales,Revenue,P&L,10,New online channel – mapping not yet confirmed
2,4000,Cost of Goods Sold,COGS,P&L,20,NaN
3,4100,Freight Costs,COGS,P&L,21,Used occasionally – please confirm if part of ...
4,5000,Salaries & Wages,Operating Expenses,P&L,30,NaN
5,5100,Rent Expense,Operating Expenses,P&L,35,NaN
6,5200,Marketing exp,Operating Expenses,P&L,40,Should this be grouped with Operating Expenses?
7,7000,Interest Expense,Financial Items,P&L,90,Financial cost – please confirm where it shoul...


In [18]:
# register dataframe as a temporary view
con.register("raw__account_mapping", df_account_mapping)

In [19]:
# Create bronze account mapping table
try:
    con.execute(
        """
        CREATE OR REPLACE TABLE bronze.account_mapping AS
        SELECT *
        FROM raw__account_mapping
        """
    )
    print("✅ Created bronze account mapping from excel")
except ValueError as error:
    print(f"Failed create bronze table, {error}")

✅ Created bronze account mapping from excel


In [20]:
# Check bronze account mapping table
con.sql(
    """
    SELECT
    *
    FROM bronze.account_mapping
    """
)

┌───────────────┬────────────────────┬────────────────────┬───────────────┬───────────┬───────────────────────────────────────────────────────────────────┐
│ AccountNumber │    AccountName     │       PLLine       │ StatementType │ SortOrder │                               Notes                               │
│     int64     │      varchar       │      varchar       │    varchar    │   int64   │                              varchar                              │
├───────────────┼────────────────────┼────────────────────┼───────────────┼───────────┼───────────────────────────────────────────────────────────────────┤
│          3000 │ Product Sales      │ Revenue            │ P&L           │        10 │ NULL                                                              │
│          3010 │ Online Sales       │ Revenue            │ P&L           │        10 │ New online channel – mapping not yet confirmed                    │
│          4000 │ Cost of Goods Sold │ COGS               │ P&L 

In [21]:
con.close()